# 01 · 데이터 확인과 그래프 구축

국가철도공단 표준데이터 3종을 읽어 **물리 네트워크 그래프**(역=노드, 운행·환승=엣지)를 만든다.

> 실행 전 `data/raw/README.md` 안내대로 운행정보 원본(약 18MB)을 `data/raw/`에 넣어야 한다.

In [ ]:
# 저장소 루트에서 실행되도록 경로 이동 (notebooks/ 안에서 열었을 때 대비)
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('작업 경로:', os.getcwd())

## 원본 데이터 훑어보기

역사정보에는 좌표·환승·운영기관이, 노선정보에는 `정거장구성`(역 순서)이,
운행정보에는 열차별 정차 시각이 들어 있다.

In [ ]:
import pandas as pd
sta = pd.read_excel('data/raw/전체_도시철도역사정보_20260630.xlsx')
lin = pd.read_excel('data/raw/전체_도시철도노선정보_20260630.xlsx')
print('역사정보', sta.shape, '| 노선정보', lin.shape)
sta.head(3)

In [ ]:
# 노선정보의 정거장구성 = 노선별 역 순서 → 인접(운행) 엣지의 근거
print(lin.loc[0, '노선명'])
print(str(lin.loc[0, '정거장구성'])[:180])

## 원본 품질 이슈 확인

정제 로직이 왜 필요한지 직접 확인한다. 자세한 근거는 `docs/README_데이터.md` 참조.

In [ ]:
# ① 역번호가 전역 고유값이 아님 — 도시 간 충돌
dup = sta[sta['역번호'].astype(str).duplicated(keep=False)]
print('중복 역번호 행수:', len(dup))
dup.sort_values('역번호')[['역번호','역사명','노선명']].head(6)

In [ ]:
# ② 동일 노선번호에 노선명이 2개인 사례
g = sta.groupby('노선번호')['노선명'].nunique()
for ln in g[g > 1].index:
    print(ln, '→', sorted(sta[sta['노선번호'] == ln]['노선명'].unique()))

## 그래프 구축 실행

`build_graph.py`가 정제·매칭·가중치 결합을 모두 수행한다.

In [ ]:
import build_graph
build_graph.main()

In [ ]:
import networkx as nx, json
G = nx.read_graphml('data/processed/network.graphml')
print(json.dumps(json.load(open('data/processed/_build_stats.json')),
                 ensure_ascii=False, indent=2))

In [ ]:
# 연결요소 = 물리적으로 분리된 도시권
for c in sorted(nx.connected_components(G), key=len, reverse=True)[:6]:
    ops = pd.Series([G.nodes[n]['운영기관'] for n in c]).value_counts()
    print(f'{len(c):4d}개 역  주요기관: {ops.index[0]}')